In [15]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib

#matplotlib.use("TkAgg")          # Or "Qt5Agg", "MacOSX", "WebAgg"
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
from params_all_cores import *
import time
import importlib
from collections import deque
import threading


# ------------------------------------------------------------
# Backend selection & WebAgg tweaks
# ------------------------------------------------------------
import os, matplotlib as mpl

if os.environ.get("DISPLAY", "") == "":          # headless session
    mpl.use("WebAgg")
    mpl.rcParams["webagg.port"] = 8988           # deterministic port
    mpl.rcParams["webagg.open_in_browser"] = False   # open ONE tab
    # (no need for 'webagg.open_in_new'; it no longer exists)
else:
    mpl.use("TkAgg")                             # local GUI
# ------------------------------------------------------------

import matplotlib.pyplot as plt

# sys.path.append('../Tools')

In [3]:
devices = samna.device.get_unopened_devices()
print(devices)

[device::DeviceInfo(serial_number=00000033, usb_bus_number=3, usb_device_address=8, logic_version=5, device_type_name=Dynapse1DevKit)]


In [4]:
#devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

In [5]:
api = model.get_dynapse1_api()
config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 
param_group_c1 = config1.chips[0].cores[1].parameter_group 
param_group_c2 = config1.chips[0].cores[2].parameter_group 
param_group_c3 = config1.chips[0].cores[3].parameter_group 

In [6]:
param_list = ["IF_AHTAU_N", "IF_AHTHR_N", "IF_AHW_P", "IF_BUF_P", "IF_DC_P", "IF_NMDA_N", "IF_RFR_N", "IF_TAU1_N", "IF_TAU2_N", "IF_THR_N", "NPDPIE_TAU_F_P", "NPDPIE_TAU_S_P", "NPDPIE_THR_F_P", "NPDPIE_THR_S_P", 
              "NPDPII_TAU_F_P", "NPDPII_TAU_S_P", "NPDPII_THR_F_P", "NPDPII_THR_S_P", "PS_WEIGHT_EXC_F_N", "PS_WEIGHT_EXC_S_N", "PS_WEIGHT_INH_F_N", "PS_WEIGHT_INH_S_N", "PULSE_PWLK_P", "R2R_P"]

In [7]:
# ----------------  stimulus: a Gaussian bump ----------------
n_pts     = 1000                 # number of samples
t_end     = 1.0                  # seconds  (→ dt = 1 ms)
t         = np.linspace(0, t_end, n_pts, endpoint=False)
x         = np.linspace(-4, 4, n_pts)
sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
#I_peak    = 30000000e-12              # 1000 pA
I_peak    = 30000000e-12              # 1000 pA

I         = gauss/gauss.max() * I_peak   # injected current (A)

# ----------------  LIF neuron parameters ----------------------
tau_m     = 20e-3                # 20 ms membrane time constant
R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
C_m       = tau_m / R_m
v_rest    = -65e-3               # -65 mV
v_reset   = -65e-3
v_thresh  = -50e-3               # spike threshold
t_ref     = 2e-3                 # 2 ms refractory period
dt        = t_end / n_pts        # simulation time-step (s)

# ----------------  simulation loop ----------------------------
v        = v_rest
next_ok  = 0.0                   # time when refractory ends
v_trace  = np.empty(n_pts)
spikes   = []

for k in range(n_pts):
    if t[k] >= next_ok:          # not in refractory
        dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
        v += dv
        if v >= v_thresh:        # spike!
            spikes.append(t[k])
            v = v_reset
            next_ok = t[k] + t_ref
    v_trace[k] = v

spike_times_all = np.array(spikes)

spike_ids = np.full(len(spikes), 1)

spikegen_ids = [(0, 1, n) for n in range(10)]

In [8]:
eventsBuffer = deque(maxlen=500)

In [9]:
def collect_spikes(sink_node, runningFlag):
    while runningFlag[0]:  # Check first element of list
        eventsBuffer.extend(sink_node.get_events())

In [10]:
import params_all_cores

importlib.reload(params_all_cores)
config1 = model.get_configuration()

pop_nr = 4

# 1)  Declare the set of bad neurons once, in (chip, core, neuron_id) format

BROKEN_NEURONS = {(0, 1, 51), (0, 1, 71), (0, 1, 80), (0, 1, 88), (0, 1, 91)}          #  ⬅️  add more here if needed

def is_ok(chip: int, core: int, nid: int) -> bool:
    """True if this physical neuron should be used."""
    return (chip, core, nid) not in BROKEN_NEURONS

In [11]:
import importlib, dynapse1utils as ut

importlib.reload(params_all_cores)

importlib.reload(ut)

config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 

p_E_E   = 1
p_mexican = 1
p_I_I   = 0 #1
p_E_I   = 0 #0.1
p_I_E   = 0 #0.4

# initiate network 
net_gen = NetworkGenerator()
net_gen.clear_network()

# create spikegens, one per ring attractor neural pop 
spikegen_ids = [(0, 0, n) for n in range(10)]
#isi_spikegen = Neuron(0, 0, 200, True)
#isi_neuron = Neuron(0, 1, 200)

spikegens = []
for spikegen_id in spikegen_ids:
    spikegens.append(Neuron(spikegen_id[0], spikegen_id[1], spikegen_id[2], True))

print(spikegens)

#spikegens.append(isi_spikegen)

# Create ring neuron populations
chip = 0
core = 1
npop = 4
NBINS = 10
offset_nr = 48

ring_pops = []
next_id = offset_nr
for _ in range(NBINS):
    pop = []
    while len(pop) < npop:
        if is_ok(chip, core, next_id):
            pop.append(Neuron(chip, core, next_id))
        next_id += 1
    ring_pops.append(pop)


# create inhibitory population that connects to all other pops
core_inh = 2
start_inh_neuron = 4
npop_inh = 4
pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]
pop_inhibitory


# SPIKEGEN CONNECTIONS: one spike-gen (index i) permanently drives ring_pops[i]
for sg, pop in zip(spikegens, ring_pops):
    for neuron in pop:
        net_gen.add_connection(sg, neuron, dyn1.Dynapse1SynType.AMPA)
    
# connect isi spikegen to isi neuron 
#net_gen.add_connection(isi_spikegen, isi_neuron, dyn1.Dynapse1SynType.AMPA)

# self excitation in each neural population in the ring: (todo determine if this is needed) 
for pop in ring_pops:
    for pre in pop:
        for post in pop:
            if pre is not post and np.random.rand() < p_E_E:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)
                                
# MEXICAN HAT CONNECTIONS
OFFSET_1 = (-1, 1)         
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors

for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)


    #  excitatory connections to third neighbors
OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                print(post)

    # todo inhibitory connections to all of the other pops
OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

# INH → EXC  (global inhibition pop to all pops in the ring)
for inh in pop_inhibitory:
    for pop in ring_pops:
        for exc in pop:
            net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

# EXC → INH  (drive the global inhibition pop from all pops in the ring)
for pop in ring_pops:
    for exc in pop:
        for inh in pop_inhibitory:
            net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)


# make a dynapse1config using the network
new_config = net_gen.make_dynapse1_configuration()

# apply the configuration
model.apply_configuration(new_config)

# Set hardware parameters
set_params(model)

fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 


monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)
    for pop in ring_pops
    for n   in pop
]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_inhibitory  
])


graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
graph.start()

# clear the buffer
sink_node.get_events()

# select the neurons to monitor
filter_node.set_neurons(monitored_neurons)

api.reset_timestamp()

ut.set_neuron_tau1(model, 0, 0, (7, 255))
ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))
ut.set_neuron_tau1(model, 0, 3, (7, 255))

time.sleep(1)

ut.set_neuron_tau1(model, 0, 0, (4, 50))
ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 50))
ut.set_neuron_tau1(model, 0, 3, (4, 50))


spike_ids_all = spike_ids

# Sort input events in time order
sort_indices = np.argsort(spike_times_all)
all_spike_times = spike_times_all[sort_indices]
all_spike_ids = spike_ids_all[sort_indices]

current_pop = {'value': 5}        # start with pop 0
spike_ids   = np.full(len(all_spike_times), current_pop['value'])

ut.set_fpga_spike_gen(
    fpga_spike_gen,
    all_spike_times,
    #all_spike_ids,
    spike_ids,
    #target_chips=[0] * len(all_spike_ids),
    target_chips=[0] * len(spike_ids),
    isi_base=900,
    repeat_mode=False)

fpga_spike_gen.start()

[C0c0s0, C0c0s1, C0c0s2, C0c0s3, C0c0s4, C0c0s5, C0c0s6, C0c0s7, C0c0s8, C0c0s9]
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50

In [12]:
import scipy.ndimage as ndi
def smooth_rates_circular(counts, sigma_bins=1.0):
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')

    return smoothed[3:-3]

In [13]:
# Start spike collection in a thread
running_flag = [True]  # Use list for mutable flag
spike_thread = threading.Thread(target=collect_spikes, args=(sink_node, running_flag))
spike_thread.daemon = True
spike_thread.start()
# animation = start_spike_visualization(eventsBuffer)

In [16]:
def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False


# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)   # moved ↓ here
fig.show()                      # ← opens ONE browser tab

# Setup raster plot in first subplot
scatter = ax_raster.scatter([], [], s=10, alpha=0.6)
xlim_max = 10
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
# Create positions for neurons (0 to 2π for the ring)
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p,            ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0  # seconds - time window for calculating rates
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]  # Convert to seconds
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
                
            # Update raster plot
            ax_raster.set_ylim(min(spikesID) - 0.5, max(spikesID) + 0.5)
            scatter.set_offsets(np.column_stack((spikesTimes, spikesID)))
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                for i, pop in enumerate(ring_pops):
                    pop_ids = [n.neuron_id for n in pop]
                    if neuron_id in pop_ids:
                        firing_rates[i] += 1
                        break
                    
            # numpy histogram for firing rate then divide by bins
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            #rate_line.set_ydata(firing_rates)
            #max_rate = max(firing_rates) if any(firing_rates > 0) else 10
            #ax_rate.set_ylim(0, max_rate * 1.2)  # Add 20% margin
            
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            ax_rate.set_ylim(0, 20)  

            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()   # keeps the websocket alive
            time.sleep(0.01)            # tiny CPU-friendly sleep

            #plt.pause(0.0001)  # Shorter pause for smoother updates
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()  # Print detailed error information
        break

/tmp/ipykernel_277576/777932402.py:76: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


[20.512494999999998, 1.4217739999999999, 1.421776, 1.471554, 1.4757939999999998, 1.686307, 1.733279, 1.7481799999999998, 1.7554429999999999, 1.7613079999999999, 1.768408, 1.794614, 1.801124, 1.811434, 1.834527, 1.874163, 1.8741949999999998, 1.92547, 1.960108, 2.034256, 2.044083, 2.115539, 2.127443, 2.136276, 2.2704, 2.299753, 2.317474, 2.3215429999999997, 2.3549599999999997, 2.355753, 2.359912, 2.478769, 2.501284, 2.6426749999999997, 2.700198, 2.700514, 2.7041459999999997, 2.753317, 2.775489, 2.870503, 2.8819429999999997, 2.8875979999999997, 2.896098, 2.896426, 2.938608, 2.957696, 3.0018909999999996, 3.086884, 3.117012, 3.278791, 3.31514, 3.3901679999999996, 3.404645, 3.478808, 3.573577, 3.5811979999999997, 3.6374579999999996, 3.689336, 3.8035599999999996, 3.889242, 3.919054, 3.980842, 3.982576, 4.1297239999999995, 4.173163, 4.187457, 4.204851, 4.226395999999999, 4.391515, 4.393323, 4.431654, 4.523146, 4.596125, 4.685112, 4.701121, 4.8382819999999995, 4.8487089999999995, 4.854244, 5.06

KeyboardInterrupt: 

In [ ]:
# To stop everything cleanly:
running_flag[0] = False
spike_thread.join(timeout=1.0)
plt.ioff()
plt.close()